# Guide v2 — YOLO + SegFormer + Depth Anything

Notebook chỉ gọi `guide_core`; không chứa thuật toán chỉ dẫn riêng. Dùng GPU T4, Internet ON để cài thư viện. Đây là **phân tích video**, không phải huấn luyện. Giữ bản desktop cũ để đối chiếu.

Tải gói `guide_kaggle_v2_models.zip` thành một Kaggle Dataset riêng, rồi Add Input. Thêm video bạn tự quay hoặc được phép dùng. Chọn GPU trước khi chạy.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['YOLO_AUTOINSTALL'] = 'false'
from pathlib import Path
print('Các thư mục input:')
for p in Path('/kaggle/input').iterdir(): print(p)
print('Các manifest, ZIP và video trong Input:')
for pattern in ('bundle_manifest.json', 'guide_kaggle_v2*.zip', '*.mp4', '*.mov', '*.MOV'):
    for p in Path('/kaggle/input').rglob(pattern): print(p)


## 1. Chọn gói đã tải lên

Đặt `BUNDLE_SOURCE` thành đường dẫn file ZIP hoặc thư mục chứa `bundle_manifest.json` được liệt kê ở trên. Chọn video ở bước 3.


In [ ]:
BUNDLE_SOURCE = Path('/kaggle/input/guide-core-v2')  # SỬA đường dẫn này
WORK = Path('/kaggle/working/guide_v2')
import zipfile, shutil, json, hashlib
assert BUNDLE_SOURCE.exists(), 'Sửa BUNDLE_SOURCE theo danh sách ở ô trên.'
WORK.mkdir(parents=True, exist_ok=True)
if BUNDLE_SOURCE.is_file():
    with zipfile.ZipFile(BUNDLE_SOURCE) as z:
        for name in z.namelist():
            dest = (WORK / name).resolve()
            assert dest.is_relative_to(WORK.resolve()) and '\\' not in name, 'Unsafe ZIP path'
        z.extractall(WORK)
else:
    manifest = json.loads((BUNDLE_SOURCE/'bundle_manifest.json').read_text())
    for item in manifest['files']:
        name = item['path']
        dest = (WORK/name).resolve()
        assert dest.is_relative_to(WORK.resolve()) and '\\' not in name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BUNDLE_SOURCE/name, dest)
    shutil.copy2(BUNDLE_SOURCE/'bundle_manifest.json', WORK/'bundle_manifest.json')
manifest = json.loads((WORK/'bundle_manifest.json').read_text())
for item in manifest['files']:
    p = (WORK/item['path']).resolve()
    assert p.is_relative_to(WORK.resolve())
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    assert h.hexdigest() == item['sha256'], f'File lỗi: {p}'
os.chdir(WORK)
print('Gói kiểm tra toàn vẹn OK; video có trong gói:', manifest.get('video'))


## 2. Cài thư viện, giữ PyTorch CUDA của Kaggle

Chạy ô này trước khi import `transformers`/`ultralytics`. Nếu đã import phiên bản khác trong phiên hiện tại, restart session rồi chạy lại. Cấu hình này đã thử trên Windows CPU; ô kiểm tra tương thích dưới đây phải đạt trên Kaggle trước khi chạy video dài.


In [ ]:
import sys, subprocess, importlib.metadata as metadata
constraints = []
for package in ('torch', 'torchvision', 'torchaudio'):
    try: constraints.append(f'{package}=={metadata.version(package)}')
    except metadata.PackageNotFoundError: pass
Path('kaggle_torch_constraints.txt').write_text('\n'.join(constraints))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
                '-c', 'kaggle_torch_constraints.txt', '-r', 'requirements-guide-core.txt'], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Bật GPU hoặc dừng để tránh nhầm benchmark CPU thành GPU.'
print('GPU:', torch.cuda.get_device_name(0))


## 3. Chọn video, bối cảnh và model

`alley`: hẻm/đường nhỏ; `sidewalk`: vỉa hè; `indoor`: hành lang trong nhà. `mirror_input=True` chỉ khi nguồn video **thực sự bị lật gương**. Không lật chỉ vì muốn ảnh trông giống camera selfie.

Mặc định một detector YOLOv8n ROD + SegFormer B0 ADE20K + Depth V2 Small. Đổi `PRIMARY` thành `rod_26` để dùng YOLO26n. `SECOND_DETECTOR=True` chỉ cho thử nghiệm so sánh, không mặc định tốt hơn.


In [ ]:
VIDEO_PATH = ''  # Dán đường dẫn MP4/MOV ở ô đầu
bundled_video = manifest.get('video')
assert VIDEO_PATH or bundled_video, 'Add Input video rồi điền VIDEO_PATH theo ô đầu.'
VIDEO = Path(VIDEO_PATH) if VIDEO_PATH else WORK / bundled_video
PRIMARY = 'rod_v8'  # 'rod_v8' hoặc 'rod_26'
SECOND_DETECTOR = False
ENVIRONMENT = 'sidewalk'
MODE = 'offline'  # 'realtime_replay' để đo bỏ khung/độ mới trên GPU
MAX_SECONDS = 6   # Đặt None để xử lý cả video
assert VIDEO.is_file(), f'Không thấy video: {VIDEO}'
assert PRIMARY in {'rod_v8', 'rod_26'}
config = json.loads(Path('configs/guide_kaggle.json').read_text())
config.update(environment=ENVIRONMENT, mode=MODE, mirror_input=False, sample_fps=10)
for m in config['models']:
    m['device'] = 'cuda:0'
    if m['role'] == 'detector': m['enabled'] = m['id'] == PRIMARY or SECOND_DETECTOR
Path('selected_config.json').write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))


## 4. Kiểm tra ba model bằng một khung hình

Ô này phải chạy xong không có `failures`. Không tiếp tục khi model chưa tải, không tương thích hoặc sai đường dẫn. Model depth trong gói đã được tải theo revision cố định.


In [ ]:
subprocess.run([sys.executable, 'tools/check_guide_environment.py', '--config', 'selected_config.json',
                '--video', str(VIDEO), '--report', '/kaggle/working/compatibility.json'], check=True)


## 5. Xuất video demo

Trong `offline`, lời cảnh báo được ghép theo thời gian video, **không phải độ trễ loa thực**. Trong `realtime_replay`, lõi mô phỏng nguồn camera chạy đúng thời gian, bỏ khung cũ và chỉ dùng kết quả còn mới. Notebook không đọc camera DroidCam qua mạng nội bộ.


In [ ]:
from datetime import datetime
RUN = Path('/kaggle/working') / ('guide_demo_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
command = [sys.executable, '-m', 'guide_core.demo', '--config', 'selected_config.json',
           '--video', str(VIDEO), '--output', str(RUN)]
if MAX_SECONDS is not None: command += ['--seconds', str(MAX_SECONDS)]
subprocess.run(command, check=True)
print('Đã xuất:', RUN/'demo.mp4')


In [ ]:
from IPython.display import Video, display
print((RUN/'report.json').read_text())
display(Video(str(RUN/'demo.mp4'), embed=True))


## 6. So sánh trên cùng khung hình (chủ động bật)

Chạy detector v8 / 26, thêm segmentation, thêm depth và thêm detector thứ hai. So sánh tốc độ này chưa đánh giá độ chính xác nếu chưa có nhãn sự kiện. Để thử 320/480/640 thêm `--sizes 320 480 640`. Mỗi thử nghiệm cần thời gian GPU; mặc định ô dưới **không chạy**.


In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    compare_dir = Path('/kaggle/working')/('guide_compare_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    subprocess.run([sys.executable, 'tools/compare_guide_models.py', '--config', 'selected_config.json',
                    '--video', str(VIDEO), '--output', str(compare_dir), '--seconds', '6', '--sizes', '480'], check=True)
    print((compare_dir/'comparison.json').read_text())


## 7. Chấm nhãn và lưu đầu ra

Trên laptop mở `tools/annotate_guide.html` để đánh dấu khoảng nguy hiểm, thông thoáng, hướng hợp lệ và hướng bị cấm. Tải JSON sau khi kiểm tra nhãn. Khi chấm, đặt `MAX_SECONDS = None` để chạy hết video. Dùng `python -m guide_core.evaluate --annotations <file.json> --run <RUN>` cho một video. Dùng `python -m guide_core.evaluate --manifest <manifest.json>` để gộp nhiều video cùng tập validation hoặc test.

Đầu ra cần tải: `demo.mp4`, `report.json`, `decisions.jsonl`, `audio_events.jsonl`, `speech_segments.json`. Save Version sau khi chạy hoàn tất. Chọn cách lưu không chạy lại nếu chỉ muốn giữ output phiên hiện tại. Không coi có file video là đã đạt mục tiêu an toàn.


In [ ]:
archive = shutil.make_archive(str(RUN), 'zip', RUN)
print('Tải tại Output:', archive)
print('Chỉ số chưa đo được sẽ là null; không thay model desktop tự động.')
